# Resumen Final del Proyecto Thoracolumbar Explicado - Colab

Este notebook consolida la historia tecnica del proyecto:

- objetivo
- datos y decisiones
- arquitectura seleccionada
- experimentos clave
- resultados comparativos
- argumentos de por que se eligio la solucion final

## Objetivo del proyecto

Construir un pipeline reproducible para identificar vertebras toracicas y lumbares
en radiografias, excluyendo la region cervical.

## 0. Preparacion de Colab

In [1]:
import os
from pathlib import Path

#try:
    #from google.colab import drive  # type: ignore
    #drive.mount('/content/drive')
#except Exception as exc:
    #print('No se detecto entorno Colab o Drive ya estaba montado:', exc)

PROJECT_ROOT = Path('./Scoliosis_Dataset')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'No existe PROJECT_ROOT={PROJECT_ROOT}. Ajusta esta ruta a tu carpeta real en Google Drive.'
    )

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

Working directory: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset


## 1. Librerias y rutas de resumen

In [2]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

CWD = Path.cwd()

ROOT_CANDIDATES = [
    CWD,
    CWD / "Scoliosis_Dataset",
    CWD / "IA-MASTER/ProyectoGrado2/Scoliosis_Dataset",
    CWD.parent / "Scoliosis_Dataset",
    Path("/content/drive/MyDrive/DataRadriografias"),
]

REQUIRED_FILES = [
    "analysis_outputs/training_runs_binary_thoracolumbar/binary_spine_test_metrics.csv",
    "analysis_outputs/training_runs_cascade_thoracolumbar_explained/thoracolumbar_partial_test_metrics.csv",
    "analysis_outputs/thoracolumbar_postprocess_anatomical_v2_explained/postprocess_v2_metric_comparison.csv",
    "analysis_outputs/visible_range_estimator_thoracolumbar_explained/clipping_metric_comparison.csv",
    "analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_clipping_metric_comparison.csv",
]


def is_valid_root(path: Path) -> bool:
    return path.exists() and all((path / rel).exists() for rel in REQUIRED_FILES)


ROOT = next((p for p in ROOT_CANDIDATES if is_valid_root(p)), None)
assert ROOT is not None, (
    "No se pudo localizar la carpeta Scoliosis_Dataset/DataRadriografias con los archivos esperados. "
    f"Directorio actual: {CWD}"
)

ROOT = ROOT.resolve()
BITACORA_PATH = ROOT / "docs" / "bitacora_proyecto_radiografias.md"
OUTPUT_DIR = ROOT / "analysis_outputs" / "final_project_summary_thoracolumbar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BINARY_TEST_PATH = ROOT / "analysis_outputs" / "training_runs_binary_thoracolumbar" / "binary_spine_test_metrics.csv"
CASCADE_TEST_PATH = ROOT / "analysis_outputs" / "training_runs_cascade_thoracolumbar_explained" / "thoracolumbar_partial_test_metrics.csv"
POST_V2_SUMMARY_PATH = ROOT / "analysis_outputs" / "thoracolumbar_postprocess_anatomical_v2_explained" / "postprocess_v2_experiment_summary.csv"
POST_V2_COMPARE_PATH = ROOT / "analysis_outputs" / "thoracolumbar_postprocess_anatomical_v2_explained" / "postprocess_v2_metric_comparison.csv"
POST_V2_PER_SAMPLE_PATH = ROOT / "analysis_outputs" / "thoracolumbar_postprocess_anatomical_v2_explained" / "postprocess_v2_per_sample_compare.csv"
RANGE_SUMMARY_PATH = ROOT / "analysis_outputs" / "visible_range_estimator_thoracolumbar_explained" / "visible_range_experiment_summary.csv"
RANGE_COMPARE_PATH = ROOT / "analysis_outputs" / "visible_range_estimator_thoracolumbar_explained" / "clipping_metric_comparison.csv"
RANGE_PER_SAMPLE_PATH = ROOT / "analysis_outputs" / "visible_range_estimator_thoracolumbar_explained" / "clipping_per_sample_compare.csv"
LAST_SUMMARY_PATH = ROOT / "analysis_outputs" / "last_visible_estimator_thoracolumbar_explained" / "last_visible_experiment_summary.csv"
LAST_COMPARE_PATH = ROOT / "analysis_outputs" / "last_visible_estimator_thoracolumbar_explained" / "last_visible_clipping_metric_comparison.csv"
LAST_PER_SAMPLE_PATH = ROOT / "analysis_outputs" / "last_visible_estimator_thoracolumbar_explained" / "last_visible_per_sample_compare.csv"
FINAL_INFERENCE_RESULTS_PATH = ROOT / "analysis_outputs" / "final_inference_pipeline_thoracolumbar" / "inference_results.csv"

print("CWD:", CWD)
print("ROOT:", ROOT)
print("BITACORA_PATH:", BITACORA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


CWD: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset
ROOT: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset
BITACORA_PATH: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/docs/bitacora_proyecto_radiografias.md
OUTPUT_DIR: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/analysis_outputs/final_project_summary_thoracolumbar


## 2. Decisiones principales del proyecto

Estas decisiones no fueron arbitrarias; surgieron del comportamiento del dataset y
de los errores observados durante la exploracion.

In [3]:
decision_rows = [
    {
        'decision': 'Excluir vertebras cervicales',
        'por_que': 'El objetivo clinico y el consejo experto enfocaron el problema en thoracic + lumbar.',
        'impacto': 'Se redujo la complejidad del problema y se concentro el aprendizaje en la region util.',
    },
    {
        'decision': 'Usar estrategia en cascada',
        'por_que': 'La columna completa es mas facil de localizar que separar directamente cada vertebra en la radiografia completa.',
        'impacto': 'El modelo multiclase trabajo dentro de una ROI mas limpia y anatomica.',
    },
    {
        'decision': 'Adoptar subset partial',
        'por_que': 'Las imagenes parciales representan un caso real y aumentan el numero de muestras disponibles.',
        'impacto': 'Se mejoro el entrenamiento, pero aparecio el problema de sobreprediccion en secuencias parciales.',
    },
    {
        'decision': 'No usar postproceso anatomico v1',
        'por_que': 'El recorte duro destruyo casi toda la segmentacion util.',
        'impacto': 'Se descarto como solucion final y quedo solo como evidencia de exploracion.',
    },
    {
        'decision': 'Mantener postproceso v2 como referencia',
        'por_que': 'Mejoro monotonia anatomica, pero apenas redujo sobreprediccion.',
        'impacto': 'Sirvio para confirmar que la solucion real debia estimar mejor el rango visible.',
    },
    {
        'decision': 'Crear visible-range estimator',
        'por_que': 'La principal falla era extender la secuencia mas alla de lo visible.',
        'impacto': 'Se logro una primera mejora real sobre raw.',
    },
    {
        'decision': 'Especializar el modelo en last_visible_idx',
        'por_que': 'El analisis demostro que la primera vertebra visible ya estaba casi resuelta y el error real estaba en la ultima.',
        'impacto': 'Se obtuvo la mejor version actual del pipeline.',
    },
]
decisions_df = pd.DataFrame(decision_rows)
display(decisions_df)

,decision,por_que,impacto
0,Excluir vertebras cervicales,El objetivo clinico y el consejo experto enfoc...,Se redujo la complejidad del problema y se con...
1,Usar estrategia en cascada,La columna completa es mas facil de localizar ...,El modelo multiclase trabajo dentro de una ROI...
2,Adoptar subset partial,Las imagenes parciales representan un caso rea...,"Se mejoro el entrenamiento, pero aparecio el p..."
3,No usar postproceso anatomico v1,El recorte duro destruyo casi toda la segmenta...,Se descarto como solucion final y quedo solo c...
4,Mantener postproceso v2 como referencia,"Mejoro monotonia anatomica, pero apenas redujo...",Sirvio para confirmar que la solucion real deb...
5,Crear visible-range estimator,La principal falla era extender la secuencia m...,Se logro una primera mejora real sobre raw.
6,Especializar el modelo en last_visible_idx,El analisis demostro que la primera vertebra v...,Se obtuvo la mejor version actual del pipeline.


## 3. Arquitectura final seleccionada

La arquitectura final no es el primer intento del proyecto, sino el resultado de
varias iteraciones guiadas por evidencia.

In [4]:
architecture_df = pd.DataFrame([
    {
        'etapa': '1. Binary spine localization',
        'entrada': 'Radiografia completa',
        'salida': 'ROI espinal',
        'razon': 'Reducir fondo y enfocar la siguiente etapa en la columna.',
    },
    {
        'etapa': '2. Multiclase thoracolumbar segmentation',
        'entrada': 'ROI espinal',
        'salida': 'Mascara vertebral T1..T12 + L1..L5',
        'razon': 'Identificar vertebras dentro de una zona ya localizada.',
    },
    {
        'etapa': '3. Last visible estimator',
        'entrada': 'ROI + features anatomicas derivadas de la prediccion multiclase',
        'salida': 'Ultima vertebra visible estimada',
        'razon': 'Corregir el principal fallo del pipeline: sobreextender la secuencia.',
    },
    {
        'etapa': '4. Anatomical clipping',
        'entrada': 'Mascara multiclase + ultima vertebra visible',
        'salida': 'Mascara final corregida',
        'razon': 'Reducir etiquetas extra y mejorar coherencia anatomica final.',
    },
])
display(architecture_df)

,etapa,entrada,salida,razon
0,1. Binary spine localization,Radiografia completa,ROI espinal,Reducir fondo y enfocar la siguiente etapa en ...
1,2. Multiclase thoracolumbar segmentation,ROI espinal,Mascara vertebral T1..T12 + L1..L5,Identificar vertebras dentro de una zona ya lo...
2,3. Last visible estimator,ROI + features anatomicas derivadas de la pred...,Ultima vertebra visible estimada,Corregir el principal fallo del pipeline: sobr...
3,4. Anatomical clipping,Mascara multiclase + ultima vertebra visible,Mascara final corregida,Reducir etiquetas extra y mejorar coherencia a...


## 4. Cronologia de experimentos relevantes

Esta tabla resume el aprendizaje del proyecto, no solo los resultados finales.

In [5]:
def fmt_metric(value: float) -> str:
    return f"{float(value):.4f}"


binary_test_df = pd.read_csv(BINARY_TEST_PATH)
cascade_test_df = pd.read_csv(CASCADE_TEST_PATH)
post_v2_summary_df = pd.read_csv(POST_V2_SUMMARY_PATH)
range_summary_df = pd.read_csv(RANGE_SUMMARY_PATH)
last_summary_df = pd.read_csv(LAST_SUMMARY_PATH)

binary_row = binary_test_df.iloc[0]
cascade_row = cascade_test_df.iloc[0]
post_v2_lookup = post_v2_summary_df.set_index("metric")["value"].to_dict()
range_lookup = range_summary_df.set_index("metric")["value"].to_dict()
last_lookup = last_summary_df.set_index("metric")["value"].to_dict()

experiment_rows = [
    {
        "stage": "Binary localization",
        "variant": "binary_spine_thoracolumbar_calibrated",
        "key_result": f"test_dice={fmt_metric(binary_row['dice'])}",
        "status": "accepted",
        "comment": f"Dice={fmt_metric(binary_row['dice'])}, IoU={fmt_metric(binary_row['iou'])}, threshold={fmt_metric(binary_row.get('selected_threshold', binary_row.get('threshold', 0.50)))}.",
    },
    {
        "stage": "Cascade multiclass",
        "variant": "partial explained improved",
        "key_result": f"test_macro_dice_fg={fmt_metric(cascade_row['macro_dice_fg'])}",
        "status": "reference",
        "comment": f"Thoracic={fmt_metric(cascade_row['macro_dice_thoracic'])}, lumbar={fmt_metric(cascade_row['macro_dice_lumbar'])}.",
    },
    {
        "stage": "Postprocess",
        "variant": "anatomical v2",
        "key_result": f"post_v2_macro_dice_fg={fmt_metric(post_v2_lookup['post_v2_macro_dice_fg'])}",
        "status": "optional",
        "comment": f"Monotonic ratio {fmt_metric(post_v2_lookup['raw_monotonic_ratio'])} -> {fmt_metric(post_v2_lookup['post_v2_monotonic_ratio'])}.",
    },
    {
        "stage": "Range estimation",
        "variant": "visible-range estimator",
        "key_result": f"pred_clip_macro_dice_fg={fmt_metric(range_lookup['pred_clip_macro_dice_fg'])}",
        "status": "reference",
        "comment": f"range_exact_acc={fmt_metric(range_lookup['range_test_exact_acc'])}, within1={fmt_metric(range_lookup['range_test_within1_acc'])}.",
    },
    {
        "stage": "Range estimation",
        "variant": "last-visible estimator",
        "key_result": f"last_pred_clip_macro_dice_fg={fmt_metric(last_lookup['last_pred_clip_macro_dice_fg'])}",
        "status": "best_current",
        "comment": f"last_exact_acc={fmt_metric(last_lookup['last_test_exact_acc'])}, within1={fmt_metric(last_lookup['last_test_within1_acc'])}.",
    },
]

experiments_df = pd.DataFrame(experiment_rows)
display(experiments_df)


,stage,variant,key_result,status,comment
0,Binary localization,binary_spine_thoracolumbar_calibrated,test_dice=0.8967,accepted,"Dice=0.8967, IoU=0.8127, threshold=0.6500."
1,Cascade multiclass,partial explained improved,test_macro_dice_fg=0.3226,reference,"Thoracic=0.2922, lumbar=0.3956."
2,Postprocess,anatomical v2,post_v2_macro_dice_fg=0.3250,optional,Monotonic ratio 0.5556 -> 1.0000.
3,Range estimation,visible-range estimator,pred_clip_macro_dice_fg=0.3205,reference,"range_exact_acc=0.2889, within1=0.4667."
4,Range estimation,last-visible estimator,last_pred_clip_macro_dice_fg=0.3324,best_current,"last_exact_acc=0.3333, within1=0.5556."


## 5. Carga de resultados exportados de las etapas finales

Aqui se leen los CSV ya generados por los notebooks del proyecto.

In [6]:
def read_metric_summary(path: Path, stage_name: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame({"metric": [], stage_name: []})
    df = pd.read_csv(path)
    if {"metric", "value"}.issubset(df.columns):
        return df.rename(columns={"value": stage_name})
    return pd.DataFrame({"metric": [], stage_name: []})


binary_df = binary_test_df.T.reset_index()
binary_df.columns = ["metric", "binary_test"]

cascade_df = cascade_test_df.T.reset_index()
cascade_df.columns = ["metric", "cascade_test"]

post_v2_df = read_metric_summary(POST_V2_SUMMARY_PATH, "post_v2")
range_df = read_metric_summary(RANGE_SUMMARY_PATH, "range_clip")
last_df = read_metric_summary(LAST_SUMMARY_PATH, "last_clip")

merged_df = binary_df.merge(cascade_df, on="metric", how="outer")
for other in [post_v2_df, range_df, last_df]:
    if not other.empty:
        merged_df = merged_df.merge(other, on="metric", how="outer")

display(merged_df.sort_values("metric").reset_index(drop=True))


,metric,binary_test,cascade_test,post_v2,range_clip,last_clip
0,dice,0.896667,NaN,NaN,NaN,NaN
1,iou,0.812689,NaN,NaN,NaN,NaN
2,last_pred_clip_macro_dice_fg,NaN,NaN,NaN,NaN,0.3323588733293916
3,last_test_exact_acc,NaN,NaN,NaN,NaN,0.3333333333333333
4,last_test_mae,NaN,NaN,NaN,NaN,2.2666666666666666
5,last_test_overprediction_rate,NaN,NaN,NaN,NaN,0.6444444444444445
6,last_test_within1_acc,NaN,NaN,NaN,NaN,0.5555555555555556
7,loss,0.339199,1.978433,NaN,NaN,NaN
8,macro_dice_fg,NaN,0.322602,NaN,NaN,NaN
9,macro_dice_lumbar,NaN,0.395596,NaN,NaN,NaN


## 6. Comparacion final de la evolucion del pipeline

Esta comparacion se centra en las variantes que realmente compitieron por entrar al
pipeline final.

In [7]:
post_v2_compare_df = pd.read_csv(POST_V2_COMPARE_PATH)
post_v2_per_sample_df = pd.read_csv(POST_V2_PER_SAMPLE_PATH)
range_compare_df = pd.read_csv(RANGE_COMPARE_PATH)
range_per_sample_df = pd.read_csv(RANGE_PER_SAMPLE_PATH)
last_compare_df = pd.read_csv(LAST_COMPARE_PATH)
last_per_sample_df = pd.read_csv(LAST_PER_SAMPLE_PATH)


def metric_from_compare(df: pd.DataFrame, metric_name: str, column: str) -> float:
    return float(df.loc[df["metric"] == metric_name, column].iloc[0])


def mean_col(df: pd.DataFrame, column: str) -> float:
    return float(df[column].mean())


final_compare_df = pd.DataFrame([
    {
        "variant": "raw_multiclass_baseline",
        "macro_dice_fg": metric_from_compare(last_compare_df, "macro_dice_fg", "raw"),
        "macro_iou_fg": metric_from_compare(last_compare_df, "macro_iou_fg", "raw"),
        "macro_dice_lumbar": metric_from_compare(last_compare_df, "macro_dice_lumbar", "raw"),
        "mean_extra_count": mean_col(last_per_sample_df, "raw_extra_count"),
        "mean_missing_count": mean_col(last_per_sample_df, "raw_missing_count"),
        "is_reference_only": False,
    },
    {
        "variant": "postprocess_v2",
        "macro_dice_fg": metric_from_compare(post_v2_compare_df, "macro_dice_fg", "value_post_v2"),
        "macro_iou_fg": metric_from_compare(post_v2_compare_df, "macro_iou_fg", "value_post_v2"),
        "macro_dice_lumbar": metric_from_compare(post_v2_compare_df, "macro_dice_lumbar", "value_post_v2"),
        "mean_extra_count": mean_col(post_v2_per_sample_df, "post_extra_count"),
        "mean_missing_count": mean_col(post_v2_per_sample_df, "post_missing_count"),
        "is_reference_only": False,
    },
    {
        "variant": "visible_range_pred_clip",
        "macro_dice_fg": metric_from_compare(range_compare_df, "macro_dice_fg", "pred_clip"),
        "macro_iou_fg": metric_from_compare(range_compare_df, "macro_iou_fg", "pred_clip"),
        "macro_dice_lumbar": metric_from_compare(range_compare_df, "macro_dice_lumbar", "pred_clip"),
        "mean_extra_count": mean_col(range_per_sample_df, "pred_extra_count"),
        "mean_missing_count": mean_col(range_per_sample_df, "pred_missing_count"),
        "is_reference_only": False,
    },
    {
        "variant": "last_visible_pred_clip",
        "macro_dice_fg": metric_from_compare(last_compare_df, "macro_dice_fg", "last_pred_clip"),
        "macro_iou_fg": metric_from_compare(last_compare_df, "macro_iou_fg", "last_pred_clip"),
        "macro_dice_lumbar": metric_from_compare(last_compare_df, "macro_dice_lumbar", "last_pred_clip"),
        "mean_extra_count": mean_col(last_per_sample_df, "last_extra_count"),
        "mean_missing_count": mean_col(last_per_sample_df, "last_missing_count"),
        "is_reference_only": False,
    },
    {
        "variant": "oracle_clip_reference",
        "macro_dice_fg": metric_from_compare(last_compare_df, "macro_dice_fg", "oracle_clip"),
        "macro_iou_fg": metric_from_compare(last_compare_df, "macro_iou_fg", "oracle_clip"),
        "macro_dice_lumbar": metric_from_compare(last_compare_df, "macro_dice_lumbar", "oracle_clip"),
        "mean_extra_count": mean_col(last_per_sample_df, "oracle_extra_count"),
        "mean_missing_count": mean_col(last_per_sample_df, "oracle_missing_count"),
        "is_reference_only": True,
    },
])

best_variant_row = (
    final_compare_df.loc[~final_compare_df["is_reference_only"]]
    .sort_values(["macro_dice_fg", "macro_iou_fg"], ascending=False)
    .iloc[0]
)
oracle_variant_row = final_compare_df.loc[final_compare_df["variant"] == "oracle_clip_reference"].iloc[0]

display(final_compare_df.sort_values("macro_dice_fg", ascending=False).reset_index(drop=True))


,variant,macro_dice_fg,macro_iou_fg,macro_dice_lumbar,mean_extra_count,mean_missing_count,is_reference_only
0,oracle_clip_reference,0.357745,0.229145,0.495506,0.000000,0.000000,True
1,last_visible_pred_clip,0.332359,0.207876,0.420505,2.244444,0.022222,False
2,postprocess_v2,0.324980,0.202220,0.400935,2.577778,1.044444,False
3,raw_multiclass_baseline,0.324869,0.201784,0.397786,3.022222,0.022222,False
4,visible_range_pred_clip,0.320511,0.198804,0.387503,3.022222,0.022222,False


## 7. Argumento de seleccion de arquitectura

Esta es la justificacion tecnica de por que se elige `last_visible_pred_clip` como
mejor version actual del pipeline.

In [8]:
raw_row = final_compare_df.loc[final_compare_df["variant"] == "raw_multiclass_baseline"].iloc[0]
best_row = best_variant_row
range_row = final_compare_df.loc[final_compare_df["variant"] == "visible_range_pred_clip"].iloc[0]
oracle_row = oracle_variant_row

rationale_df = pd.DataFrame([
    {
        "argumento": "Mejor equilibrio entre mejora y estabilidad",
        "detalle": (
            f"La mejor variante real fue {best_row['variant']} con macro_dice_fg={best_row['macro_dice_fg']:.4f}, "
            f"superando a raw={raw_row['macro_dice_fg']:.4f}."
        ),
    },
    {
        "argumento": "Oracle se mantiene como techo de referencia",
        "detalle": (
            f"oracle_clip_reference alcanzo macro_dice_fg={oracle_row['macro_dice_fg']:.4f}, "
            "pero no se considera una variante desplegable del pipeline, sino una referencia superior."
        ),
    },
    {
        "argumento": "Ataca el cuello de botella correcto",
        "detalle": (
            "El proyecto siguio mostrando que la mayor mejora real aparece al recortar por ultima vertebra visible, "
            "no solo por reglas anatomicas generales."
        ),
    },
    {
        "argumento": "Mejora concreta en region lumbar",
        "detalle": (
            f"La mejor variante real alcanzo macro_dice_lumbar={best_row['macro_dice_lumbar']:.4f}, "
            f"frente a raw={raw_row['macro_dice_lumbar']:.4f}."
        ),
    },
    {
        "argumento": "Reduccion de etiquetas extra",
        "detalle": (
            f"mean_extra_count bajo de {raw_row['mean_extra_count']:.2f} a {best_row['mean_extra_count']:.2f}."
        ),
    },
    {
        "argumento": "Clipping por rango completo no basto",
        "detalle": (
            f"visible_range_pred_clip quedo en macro_dice_fg={range_row['macro_dice_fg']:.4f}, "
            f"por debajo de {best_row['variant']}."
        ),
    },
])

display(rationale_df)


,argumento,detalle
0,Mejor equilibrio entre mejora y estabilidad,La mejor variante real fue last_visible_pred_c...
1,Oracle se mantiene como techo de referencia,oracle_clip_reference alcanzo macro_dice_fg=0....
2,Ataca el cuello de botella correcto,El proyecto siguio mostrando que la mayor mejo...
3,Mejora concreta en region lumbar,La mejor variante real alcanzo macro_dice_lumb...
4,Reduccion de etiquetas extra,mean_extra_count bajo de 3.02 a 2.24.
5,Clipping por rango completo no basto,visible_range_pred_clip quedo en macro_dice_fg...


## 8. Recomendaciones finales del proyecto

In [9]:
best_variant_name = str(best_row["variant"])

recommendations_df = pd.DataFrame([
    {
        "tipo": "Pipeline final",
        "recomendacion": (
            f"Usar binary -> multiclass -> {best_variant_name} como referencia operativa actual, "
            "segun las metricas reales exportadas hasta el notebook 08."
        ),
    },
    {
        "tipo": "Uso practico",
        "recomendacion": (
            "Apoyarse en el notebook final de inferencia y revisar visualmente los casos donde el numero final de etiquetas "
            "baja de forma importante respecto a la prediccion cruda."
        ),
    },
    {
        "tipo": "Validacion futura",
        "recomendacion": (
            "Evaluar el pipeline sobre un conjunto externo o prospectivo para confirmar si la mejora en coherencia anatomica "
            "tambien se sostiene fuera del test actual."
        ),
    },
    {
        "tipo": "Mejora futura",
        "recomendacion": (
            "Usar oracle_clip_reference solo como techo de referencia y seguir optimizando la variante real last_visible_pred_clip, "
            "fortaleciendo sobre todo la region toracica."
        ),
    },
])

display(recommendations_df)


,tipo,recomendacion
0,Pipeline final,Usar binary -> multiclass -> last_visible_pred...
1,Uso practico,Apoyarse en el notebook final de inferencia y ...
2,Validacion futura,Evaluar el pipeline sobre un conjunto externo ...
3,Mejora futura,Usar oracle_clip_reference solo como techo de ...


## 9. Exportacion del resumen final

Esta celda guarda tablas utiles para el documento final.

In [10]:
decisions_path = OUTPUT_DIR / "final_decisions_table.csv"
architecture_path = OUTPUT_DIR / "final_architecture_table.csv"
experiments_path = OUTPUT_DIR / "final_experiments_table.csv"
compare_path = OUTPUT_DIR / "final_pipeline_comparison.csv"
rationale_path = OUTPUT_DIR / "final_rationale_table.csv"
recommendations_path = OUTPUT_DIR / "final_recommendations_table.csv"
merged_metrics_path = OUTPUT_DIR / "final_stage_metric_summary.csv"
inference_results_copy_path = OUTPUT_DIR / "final_inference_results_reference.csv"

decisions_df.to_csv(decisions_path, index=False)
architecture_df.to_csv(architecture_path, index=False)
experiments_df.to_csv(experiments_path, index=False)
final_compare_df.to_csv(compare_path, index=False)
rationale_df.to_csv(rationale_path, index=False)
recommendations_df.to_csv(recommendations_path, index=False)
merged_df.to_csv(merged_metrics_path, index=False)

if FINAL_INFERENCE_RESULTS_PATH.exists():
    pd.read_csv(FINAL_INFERENCE_RESULTS_PATH).to_csv(inference_results_copy_path, index=False)

if BITACORA_PATH.exists():
    print("Bitacora disponible en:", BITACORA_PATH)

print("Guardado:", decisions_path)
print("Guardado:", architecture_path)
print("Guardado:", experiments_path)
print("Guardado:", compare_path)
print("Guardado:", rationale_path)
print("Guardado:", recommendations_path)
print("Guardado:", merged_metrics_path)
if FINAL_INFERENCE_RESULTS_PATH.exists():
    print("Guardado:", inference_results_copy_path)


Guardado: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/analysis_outputs/final_project_summary_thoracolumbar/final_decisions_table.csv
Guardado: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/analysis_outputs/final_project_summary_thoracolumbar/final_architecture_table.csv
Guardado: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/analysis_outputs/final_project_summary_thoracolumbar/final_experiments_table.csv
Guardado: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/analysis_outputs/final_project_summary_thoracolumbar/final_pipeline_comparison.csv
Guardado: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/analysis_outputs/final_project_summary_thoracolumbar/final_rationale_table.csv
Guardado: /Users/camilo/Documents/WorkSpace/IA-MASTER/ProyectoGrado2/Scoliosis_Dataset/analysis_outputs/final_project_summary_thoracolumbar/final_recommendations